# Chapter 15：Triton Performance Mental Model

本章对齐 CS336 / LLM systems 的性能推理能力。kernel 快慢不取决于代码看起来长不长，而取决于数据移动、计算量、数据复用以及硬件的 memory/compute 上限。

In [ ]:
from dataclasses import dataclass
import math
import time
import torch
import torch.nn.functional as F

try:
    import triton.testing
except ModuleNotFoundError:
    triton = None

## FLOPs、bytes moved 与 arithmetic intensity

`intensity = FLOPs / bytes_moved`。vector add 每元素约 1 次加法，却要读 x、读 y、写 z，通常 memory-bound。Matmul 约有 `2*M*N*K` FLOPs，并能通过 tiling 重用数据，因此更容易 compute-bound。

Softmax 和 LayerNorm 的 FLOP 数只做教学估算；exp、rsqrt 等操作不能简单等同于一次普通加法。

In [ ]:
@dataclass(frozen=True)
class Estimate:
    estimated_flops: float
    estimated_bytes: float
    arithmetic_intensity: float


def get_dtype_bytes(dtype: torch.dtype) -> int:
    return torch.empty((), dtype=dtype).element_size()


def _estimate(flops: float, moved_bytes: float) -> Estimate:
    return Estimate(flops, moved_bytes, flops / moved_bytes if moved_bytes else 0.0)


def estimate_vector_add(n: int, dtype_bytes: int = 4) -> Estimate:
    return _estimate(n, 3 * n * dtype_bytes)


def estimate_matmul(m: int, n: int, k: int, dtype_bytes: int = 2) -> Estimate:
    flops = 2 * m * n * k
    ideal_bytes = (m * k + k * n + m * n) * dtype_bytes
    return _estimate(flops, ideal_bytes)


def estimate_softmax(m: int, n: int, dtype_bytes: int = 4) -> Estimate:
    # Approximation: max, subtract, exp, sum, and divide per element.
    return _estimate(5 * m * n, 2 * m * n * dtype_bytes)


def estimate_layernorm(m: int, n: int, dtype_bytes: int = 4) -> Estimate:
    # Approximation: mean, variance, normalize, scale, and bias.
    return _estimate(8 * m * n, (2 * m * n + 2 * n) * dtype_bytes)

## 格式化和理论表

Matmul bytes 是理想最小读写量，实际 cache miss 和 tile 调度可能增加流量。Fused softmax/LayerNorm 避免中间 tensor，而 naive 多 kernel 实现可能多次读写 HBM。

In [ ]:
def format_bytes(num_bytes: float) -> str:
    units = ("B", "KiB", "MiB", "GiB", "TiB")
    value = float(num_bytes)
    for unit in units:
        if abs(value) < 1024 or unit == units[-1]:
            return f"{value:.2f} {unit}"
        value /= 1024
    raise AssertionError("unreachable")


def format_flops(num_flops: float) -> str:
    units = ((1e12, "TFLOP"), (1e9, "GFLOP"), (1e6, "MFLOP"), (1e3, "KFLOP"))
    for scale, unit in units:
        if num_flops >= scale:
            return f"{num_flops / scale:.2f} {unit}"
    return f"{num_flops:.2f} FLOP"


def run_estimation_table() -> None:
    rows = [
        ("vector add fp32", estimate_vector_add(1_000_000, 4)),
        ("matmul fp16", estimate_matmul(1024, 1024, 1024, 2)),
        ("softmax fp32", estimate_softmax(1024, 1024, 4)),
        ("layernorm fp32", estimate_layernorm(1024, 1024, 4)),
    ]
    print(f"{'operation':<20} {'FLOPs':>14} {'bytes moved':>16} {'FLOPs/byte':>12}")
    print("-" * 66)
    for name, estimate in rows:
        print(f"{name:<20} {format_flops(estimate.estimated_flops):>14} {format_bytes(estimate.estimated_bytes):>16} {estimate.arithmetic_intensity:12.2f}")
    print("\nLow intensity often suggests memory-bound behavior; high intensity makes compute limits more likely.")

run_estimation_table()

## 实际 benchmark 与 achieved 指标

benchmark 使用 `triton.testing.do_bench` 正确处理 CUDA 异步执行。`GB/s = estimated_bytes / time`，`TFLOP/s = estimated_flops / time`。这些只是粗略估算：PyTorch 可能调用高度优化 kernel，cache 和中间结果未完全建模，小 shape 还可能被 launch overhead 支配。

In [ ]:
def benchmark(fn, warmup: int = 20, rep: int = 100) -> float:
    if triton is None:
        raise RuntimeError("Triton is required for CUDA benchmarking")
    return float(triton.testing.do_bench(fn, warmup=warmup, rep=rep, return_mode="median"))


def run_actual_benchmarks() -> None:
    if not torch.cuda.is_available():
        print("Skipping actual benchmarks because CUDA is not available.")
        return
    if triton is None:
        print("Skipping actual benchmarks because Triton is not installed.")
        return

    device = torch.device("cuda")
    x = torch.randn(1_000_000, device=device)
    y = torch.randn_like(x)
    a = torch.randn(1024, 1024, device=device, dtype=torch.float16)
    b = torch.randn_like(a)
    matrix = torch.randn(1024, 1024, device=device)
    weight = torch.ones(1024, device=device)
    bias = torch.zeros(1024, device=device)
    operations = [
        ("vector add", lambda: x + y, estimate_vector_add(x.numel(), get_dtype_bytes(x.dtype))),
        ("matmul", lambda: torch.matmul(a, b), estimate_matmul(1024, 1024, 1024, get_dtype_bytes(a.dtype))),
        ("softmax", lambda: torch.softmax(matrix, dim=1), estimate_softmax(1024, 1024, get_dtype_bytes(matrix.dtype))),
        ("layernorm", lambda: F.layer_norm(matrix, (1024,), weight, bias), estimate_layernorm(1024, 1024, get_dtype_bytes(matrix.dtype))),
    ]
    print(f"\n{'operation':<14} {'time ms':>10} {'approx GB/s':>14} {'approx TFLOP/s':>16}")
    print("-" * 58)
    for name, fn, estimate in operations:
        milliseconds = benchmark(fn)
        seconds = milliseconds / 1000
        gb_per_second = estimate.estimated_bytes / seconds / 1e9
        tflops = estimate.estimated_flops / seconds / 1e12
        print(f"{name:<14} {milliseconds:10.3f} {gb_per_second:14.2f} {tflops:16.3f}")
    print("These achieved numbers are rough estimates; cache, intermediates, implementation details, and launch overhead are simplified.")

In [ ]:
run_actual_benchmarks()

## 小结

- 低 arithmetic intensity 更容易受 memory bandwidth 限制。
- 高 intensity 且数据复用良好的 matmul 更容易接近 compute 上限。
- 优化 vector add 和 matmul 的思路不同：前者重视减少流量/融合，后者重视 tile、复用和 tensor core 吞吐。

**练习**：将 matmul 的 M/N/K 各翻倍，观察 FLOPs、理想 bytes 和 intensity 如何变化。